In [17]:
!pip install pyreadstat tqdm

In [18]:
import os
import json
import hashlib
import requests
import zipfile
import time
import pandas as pd
from tqdm import tqdm
from datetime import datetime

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"

RAW_DIR = f"{ROOT}/01BasesDatosPrimarias"
LOG_DIR = f"{ROOT}/05Resultados/logs"

In [21]:
base_url = "https://proyectos.inei.gob.pe/iinei/srienaho/descarga/SPSS/976-Modulo{}.zip"

modules = []

for i in range(1941, 1963):  # 1962 inclusive
    modules.append({
        "module": f"Modulo{i}",
        "url": base_url.format(i)
    })



In [22]:
def sha256_file(path):
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

In [23]:
def download_file(url, output_path, retries=3):
    if os.path.exists(output_path):
        print(f"Already exists: {output_path}")
        return

    for attempt in range(retries):
        try:
            r = requests.get(url, stream=True, timeout=30)
            r.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            print(f"Downloaded: {output_path}")
            return

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)

    raise Exception(f"Failed to download: {url}")

In [24]:
def extract_zip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

In [25]:
manifest = []
catalog = []
log_lines = []

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

In [26]:
for m in modules:

    module_name = m["module"]
    url = m["url"]

    module_folder = os.path.join(RAW_DIR, module_name)
    os.makedirs(module_folder, exist_ok=True)

    zip_path = os.path.join(module_folder, f"{module_name}.zip")

    log_lines.append(f"Processing {module_name}")

    # Download
    try:
        download_file(url, zip_path)
    except Exception as e:
        log_lines.append(f"FAILED download {module_name}: {str(e)}")
        continue

    zip_hash = sha256_file(zip_path)
    zip_size = os.path.getsize(zip_path)

    # Extract
    extract_folder = os.path.join(module_folder, "extracted")
    os.makedirs(extract_folder, exist_ok=True)

    extract_zip(zip_path, extract_folder)

    extracted_files = []

    for root, _, files in os.walk(extract_folder):
        for file in files:
            full_path = os.path.join(root, file)

            file_hash = sha256_file(full_path)
            file_size = os.path.getsize(full_path)

            ext = file.split(".")[-1]

            extracted_files.append(file)

            catalog.append({
                "modulo": module_name,
                "archivo": file,
                "extension": ext,
                "file_role": "raw_extracted",
                "MB": file_size / (1024*1024),
                "sha256": file_hash
            })

    manifest.append({
        "module_id": module_name,
        "selected_download_format": "SPSS ZIP",
        "official_zip_file": zip_path,
        "zip_sha256": zip_hash,
        "zip_size_bytes": zip_size,
        "extracted_files": extracted_files,
        "status": "success",
        "timestamp": timestamp
    })

    log_lines.append(f"SUCCESS {module_name}")

Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1941/Modulo1941.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1942/Modulo1942.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1943/Modulo1943.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1944/Modulo1944.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1945/Modulo1945.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1946/Modulo1946.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1947/Modulo1947.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1948/Modulo1948.zip
Already exists: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1949/Modulo1949.zip
Already exists: /content/drive/MyDriv

In [27]:
manifest_path = os.path.join(RAW_DIR, f"ENARES_2024_STAGE1_manifest_{timestamp}.json")

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("Manifest saved:", manifest_path)

Manifest saved: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/ENARES_2024_STAGE1_manifest_20260515_160941.json


In [28]:
log_path = os.path.join(LOG_DIR, f"ENARES_2024_STAGE1_log_ingesta_{timestamp}.txt")

with open(log_path, "w") as f:
    f.write("\n".join(log_lines))

print("Log saved:", log_path)

Log saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE1_log_ingesta_20260515_160941.txt


In [29]:
catalog_df = pd.DataFrame(catalog)

catalog_path = os.path.join(LOG_DIR, f"ENARES_2024_STAGE1_catalogo_modulos.csv")

catalog_df.to_csv(catalog_path, index=False)

print("Catalogue saved:", catalog_path)

Catalogue saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE1_catalogo_modulos.csv
